In [13]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, cohen_kappa_score
from imblearn.over_sampling import SMOTE

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import random


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
seed_everything(seed=42)


### 参数设置

In [14]:
IMG_SIZE = 448
BATCH_SIZE = 128
data_dir = "../data/aptos/"
exp_msg = "bottleneck-classfier"
current_notebook = "classify_head_effb4.ipynb"
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
weight_dir = '/home/gsintern/gs-lts/Template/d2l-main/avote_all_train/exp_effb4_regression/exp2/'
weight_path = os.path.join(weight_dir, 'best_model_reg.pth')
base_output_dir = os.path.join(weight_dir, 'classify_head')

if_manifold_mixup = False
if if_manifold_mixup:
    p_mixup = 0.5  # 只有50%的概率触发 Mixup
    alpha = 0.4    # 减小 alpha，使合成样本更接近原始样本

lr=9e-5
scheduler_patience = 5
scheduler_facotr = 0.5
epochs = 40
patience = 10
num_classes = 5

devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]

os.makedirs(base_output_dir, exist_ok=True)
# 自动获取下一个 exp{n} 序号
existing_exps = [d for d in os.listdir(base_output_dir) if d.startswith('exp') and d[3:].isdigit()]
next_id = max([int(d[3:]) for d in existing_exps], default=0) + 1
exp_dir = os.path.join(base_output_dir, f"exp{next_id}")
os.makedirs(exp_dir, exist_ok=True)
print(f"当前实验结果将保存至: {exp_dir}")

当前实验结果将保存至: /home/gsintern/gs-lts/Template/d2l-main/avote_all_train/exp_effb4_regression/exp2/classify_head/exp1


### 数据读取

In [15]:
import numpy as np
import pandas as pd
from PIL import Image
from joblib import Parallel, delayed # 并行加速

image_dir = os.path.join(data_dir, "image")
X_cache_path = os.path.join(data_dir, f"X_preprocessed_{IMG_SIZE}.npy")
y_cache_path = os.path.join(data_dir, f"y_preprocessed_{IMG_SIZE}.npy")
csv_path = os.path.join(data_dir, "train.csv")
train_data = pd.read_csv(csv_path)

# 返回 PIL.Image
def load_one_image(image_id):
    img_path = os.path.join(image_dir, f"{image_id}.png")
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return img


def load_raw_data():
    if os.path.exists(X_cache_path) and os.path.exists(y_cache_path):
        print("检测到缓存文件，正在快速加载...")
        X = np.load(X_cache_path)
        y = np.load(y_cache_path)
    else:
        print("首次运行，正在处理图片（可能较慢）...")
        print("开启多进程并行读取...")
        X_list = Parallel(n_jobs=-1)(
            delayed(load_one_image)(image_id) for image_id in train_data['id_code']
        )
        X = np.array(X_list)
        y = train_data['diagnosis'].values
        # 保存原始图片
        np.save(X_cache_path, X)
        np.save(y_cache_path, y)
    return X, y
X_train, y_train = load_raw_data()

检测到缓存文件，正在快速加载...


In [16]:
import numpy as np
import pandas as pd
from PIL import Image
from joblib import Parallel, delayed # 并行加速
data_dir = "../data/aptos-2019/"
image_dir = os.path.join(data_dir, "val_images")
X_cache_path = os.path.join(data_dir, f"X_preprocessed_{IMG_SIZE}.npy")
y_cache_path = os.path.join(data_dir, f"y_preprocessed_{IMG_SIZE}.npy")
csv_path = os.path.join(data_dir, "valid.csv")
train_data = pd.read_csv(csv_path)

# 返回 PIL.Image
def load_one_image(image_id):
    img_path = os.path.join(image_dir, f"{image_id}.png")
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return img


def load_raw_data():
    if os.path.exists(X_cache_path) and os.path.exists(y_cache_path):
        print("检测到缓存文件，正在快速加载...")
        X = np.load(X_cache_path)
        y = np.load(y_cache_path)
    else:
        print("首次运行，正在处理图片（可能较慢）...")
        print("开启多进程并行读取...")
        X_list = Parallel(n_jobs=-1)(
            delayed(load_one_image)(image_id) for image_id in train_data['id_code']
        )
        X = np.array(X_list)
        y = train_data['diagnosis'].values
        # 保存原始图片
        np.save(X_cache_path, X)
        np.save(y_cache_path, y)
    return X, y

X_val, y_val = load_raw_data()

检测到缓存文件，正在快速加载...


### 创建原始图片数据集

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from imblearn.over_sampling import SMOTE

class DRDataset(Dataset):
    # images 应该是未归一化的图片
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        image, label = self.images[idx], self.labels[idx]
        image = Image.fromarray(image)
        if self.transform:
            image = self.transform(image)
        # 回归任务中标签需为浮点数
        return image, torch.tensor(label, dtype=torch.float) 
# 创建专门用于特征提取的 Dataset (无增强)
extract_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

print(f"训练集形状: {X_train.shape}")
print(f"验证集形状: {X_val.shape}")

train_extract_loader = DataLoader(DRDataset(X_train, y_train, extract_tf), batch_size=64, shuffle=True, num_workers=16)
val_extract_loader = DataLoader(DRDataset(X_val, y_val, extract_tf), batch_size=64, shuffle=False, num_workers=16)

训练集形状: (2930, 448, 448, 3)
验证集形状: (366, 448, 448, 3)


### 提取特征

In [18]:
def extract_features(loader, model):
    model.eval()
    features, labels = [], []
    # 获取基础模型（处理 DataParallel 包装的情况）
    base_model = model.module if hasattr(model, 'module') else model
    # 根据模型架构动态构建特征提取器
    if hasattr(base_model, 'features'):
        # 适用于 EfficientNet 系列
        feature_extractor = nn.Sequential(base_model.features, base_model.avgpool)
    elif hasattr(base_model, 'stem') and hasattr(base_model, 'trunk_output'):
        # 适用于 RegNet 系列
        feature_extractor = nn.Sequential(base_model.stem, base_model.trunk_output, base_model.avgpool)
    else:
        # 通用方案：取倒数第一层（分类头）之前的所有层
        feature_extractor = nn.Sequential(*list(base_model.children())[:-1])
    with torch.no_grad():
        for imgs, lbls in tqdm(loader, desc="特征提取"):
            # 提取特征并展平
            feat = feature_extractor(imgs.to(devices[0]))
            feat = torch.flatten(feat, 1) 
            features.append(feat.cpu().numpy())
            labels.append(lbls.numpy())
    return np.vstack(features), np.concatenate(labels).astype(int)

In [19]:
def create_effb4_regressor():
    model = models.efficientnet_b4(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, 1 + num_classes)
    return model
# 加载回归模型并移除分类头
model = create_effb4_regressor()
model.load_state_dict(torch.load(weight_path))
model.to(devices[0])
model.eval()

X_train_feat, y_train_feat = extract_features(train_extract_loader, model)
X_val_feat, y_val_feat = extract_features(val_extract_loader, model)

/tmp/ipykernel_2572484/2500596066.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weight_path))
特征提取: 100%|██████████| 6/6 [00:02<00:00,

### 特征级SMOTE + 创建特征数据集

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_feat, y_train_feat)
train_feat_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_res), torch.LongTensor(y_train_res)), batch_size=BATCH_SIZE, shuffle=True)
val_feat_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val_feat), torch.LongTensor(y_val_feat)), batch_size=BATCH_SIZE)

In [ ]:
from collections import Counter
print(f"过采样前样本数: {len(y_train_feat)}, 各类分布: {Counter(y_train_feat)}")
print(f"过采样后样本数: {len(y_train_res)}, 各类分布: {Counter(y_train_res)}")

过采样前样本数: 2490, 各类分布: Counter({0: 1218, 2: 687, 1: 255, 4: 199, 3: 131})
过采样后样本数: 6090, 各类分布: Counter({0: 1218, 2: 1218, 4: 1218, 3: 1218, 1: 1218})


### 定义新的 ResMLP 分类头

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.LeakyReLU(0.2), # LeakyReLU 避免神经元死亡
            nn.Dropout(0.3),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.relu = nn.LeakyReLU(0.2)

    def forward(self, x):
        return self.relu(x + self.block(x))

class AdvancedMLP(nn.Module):
    def __init__(self, in_features=1792, num_classes=5):
        super().__init__()
        self.input_layer = nn.Sequential(
            nn.Linear(in_features, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2)
        )
        # 两个残差块
        self.res1 = ResBlock(1024)
        self.res2 = ResBlock(1024)
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, num_classes)
        )
    def forward(self, x):
        x = self.input_layer(x)
        x = self.res1(x)
        x = self.res2(x)
        return self.classifier(x)

### Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        :param alpha: 类别权重张量，形状为 [num_classes]
        :param gamma: 调节因子，通常设为 2.0，越大则模型越关注难分类样本
        :param reduction: 损失聚合方式
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss) # 模型预测属于正确类别的概率
        # Focal Loss: FL = (1-pt)^gamma * CE
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


### 训练循环

In [ ]:

# 训练 MLP
head_model = AdvancedMLP().to(devices[0])
optimizer = optim.Adam(head_model.parameters(), lr=lr)
# criterion = nn.CrossEntropyLoss()
criterion = FocalLoss(alpha=None, gamma=1.5).to(devices[0])
if len(devices) > 1:
    head_model = nn.DataParallel(head_model, device_ids=range(len(devices)))

print("开始训练 MLP 分类头...")
best_val_acc = 0
counter = 0
for epoch in range(epochs):
    head_model.train()
    train_loss, train_correct = 0.0, 0
    for f, l in train_feat_loader:
        f, l = f.to(devices[0]), l.to(devices[0])
        optimizer.zero_grad()

        outputs = head_model(f)
        loss = criterion(outputs, l)
        train_correct += (outputs.argmax(1) == l).sum().item()

        loss.backward()
        optimizer.step()
        train_loss += loss.item() * f.size(0)


    train_acc = train_correct / len(train_feat_loader.dataset)
    avg_train_loss = train_loss / len(train_feat_loader.dataset)

    head_model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for f_val, l_val in val_feat_loader:
            f_val, l_val = f_val.to(devices[0]), l_val.to(devices[0])
            
            outputs_val = head_model(f_val)
            loss_val = criterion(outputs_val, l_val)
            
            val_loss += loss_val.item() * f_val.size(0)
            val_correct += (outputs_val.argmax(1) == l_val).sum().item()

    val_acc = val_correct / len(val_feat_loader.dataset)
    avg_val_loss = val_loss / len(val_feat_loader.dataset)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        counter = 0 
        # 考虑 DataParallel 的保存方式
        state_dict = head_model.module.state_dict() if hasattr(head_model, 'module') else head_model.state_dict()
        save_path = os.path.join(exp_dir, "best_mlp_head.pth")
        torch.save(state_dict, save_path)
        print(f"--> 发现更优模型，已保存至: {save_path}")
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

print(f"训练完成，最佳验证准确率: {best_val_acc:.4f}")

开始训练 MLP 分类头...
Epoch 1/1 | Train Loss: 0.1721 | Acc: 0.8810 | Val Loss: 0.1386 | Val Acc: 0.9000
--> 发现更优模型，已保存至: /home/gsintern/gs-lts/Template/d2l-main/avote_all_train/exp_effb4_regression/exp2/classify_head/exp1/best_mlp_head.pth
训练完成，最佳验证准确率: 0.9000


### 记录实验结果

In [ ]:
print(f"best_val_acc: {best_val_acc}\n")
# 计算最佳准确率并格式化文件名
save_filename = f"best_val_acc{best_val_acc:.4f}.txt"
file_path = os.path.join(exp_dir, save_filename)
print(f"正在将历史指标写入: {file_path}")

with open(file_path, "w", encoding='utf-8') as f:
    pass
print("记录写入完成。")

import shutil
if os.path.exists(current_notebook):
    shutil.copy(current_notebook, os.path.join(exp_dir, f"{current_notebook}_{exp_msg}.ipynb"))
    print(f"代码已备份至: {os.path.join(exp_dir, current_notebook)}_{exp_msg}.ipynb")
else:
    print(f"警告: 未找到文件 {current_notebook}，备份失败。请检查文件名是否正确。")
print(f"当前实验结果将保存至: {exp_dir}")

best_val_acc: 0.9

正在将历史指标写入: /home/gsintern/gs-lts/Template/d2l-main/avote_all_train/exp_effb4_regression/exp2/classify_head/exp1/best_val_acc0.9000.txt
记录写入完成。
代码已备份至: /home/gsintern/gs-lts/Template/d2l-main/avote_all_train/exp_effb4_regression/exp2/classify_head/exp1/classify_head_effb4.ipynb_bottleneck-classfier.ipynb
当前实验结果将保存至: /home/gsintern/gs-lts/Template/d2l-main/avote_all_train/exp_effb4_regression/exp2/classify_head/exp1
